# Tool calling — the description is the prompt

**Session 6 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Two tools, and the only thing telling the model which to use is the `description` field. Swap
the good descriptions for vague ones and watch routing accuracy fall — the description *is* a
prompt, and you should test it like one.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import ollama  # tool calling needs a tool-capable model (llama3.2:3b is)
from utils import SMALL_MODEL, BIG_MODEL
from eval import load_cases


### Worked example

`tool_routing.jsonl` has 14 prompts labelled `weather`, `currency`, or `none`. We give the
model both tools and score how often it picks the right one (or none), with **good**
descriptions and with **vague** ones.

In [ ]:
cases = load_cases("../eval/datasets/tool_routing.jsonl")

def tools(weather_desc, currency_desc):
    return [
        {"type": "function", "function": {
            "name": "get_weather", "description": weather_desc,
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}},
                           "required": ["city"]}}},
        {"type": "function", "function": {
            "name": "convert_currency", "description": currency_desc,
            "parameters": {"type": "object", "properties": {
                "amount": {"type": "number"}, "from_ccy": {"type": "string"},
                "to_ccy": {"type": "string"}}, "required": ["amount", "from_ccy", "to_ccy"]}}},
    ]

GOOD_W = ("Get current weather or the short-term forecast for a city: temperature, rain, "
          "snow, wind, or whether to take an umbrella.")
GOOD_C = ("Convert an amount of money from one currency to another using current exchange "
          "rates.")
VAGUE_W = "Weather tool."
VAGUE_C = "Currency tool."

NAME_TO_LABEL = {"get_weather": "weather", "convert_currency": "currency"}

def route(prompt, tspec, model):
    r = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}],
                    tools=tspec, options={"temperature": 0})
    calls = r["message"].get("tool_calls") or []
    return NAME_TO_LABEL.get(calls[0]["function"]["name"], "none") if calls else "none"

def routing_accuracy(tspec, model):
    return sum(route(c["input"], tspec, model) == c["expected"] for c in cases), len(cases)

for model in (SMALL_MODEL, BIG_MODEL):
    try:
        g = routing_accuracy(tools(GOOD_W, GOOD_C), model)
        v = routing_accuracy(tools(VAGUE_W, VAGUE_C), model)
        print(f"{model:24}  good desc {g[0]}/{g[1]}   vague desc {v[0]}/{v[1]}")
    except Exception as e:
        print(f"{model:24}  skipped: {type(e).__name__}")


### Which prompts does the vague version get wrong?

The `none` cases ("what is 15% of 240?") are where a vague description hurts most: the model
has nothing telling it *not* to reach for a tool.

In [ ]:
vague = tools(VAGUE_W, VAGUE_C)
for c in cases:
    got = route(c["input"], vague, SMALL_MODEL)
    flag = "  <-- wrong" if got != c["expected"] else ""
    print(f"  want {c['expected']:8} got {got:8} | {c['input']}{flag}")


## Your turn - vary the example

1. Add a second tool (e.g. `currency_convert`) and prompts that should route to each.
2. Degrade the GOOD description one word at a time - where does routing break?
3. Add an ambiguous prompt ("how's Paris?"). Which tool fires, and is that right?
